# 📄 Universal Document → Image → LLM Pipeline

Provide **any** document (PDF, JPEG, PNG, XLSX, CSV, DOC, DOCX, PPT, PPTX, TIFF, BMP, WEBP, etc.)  
and this notebook will:
1. **Detect** the file type automatically
2. **Convert** it to image(s) and save to `converted/` folder
3. **Send** the converted images to Gemma 4 LLM along with your prompt

### Supported Conversions
| Source Format | Conversion Method |
|---|---|
| JPEG, PNG, JPG | Copy / convert to JPEG |
| BMP, TIFF, WEBP, GIF, HEIC | PIL → JPEG |
| PDF | PyMuPDF → page images |
| XLSX, XLS, CSV | pandas + matplotlib → table image |
| DOC, DOCX, PPT, PPTX, ODT, RTF | LibreOffice → PDF → page images |

## 1. Install Dependencies

In [ ]:
!pip install -q transformers torch PyMuPDF Pillow pandas matplotlib openpyxl

## 2. Load Gemma 4 Model

In [ ]:
MODEL_ID = "google/gemma-4-E2B-it"  # @param ["google/gemma-4-E2B-it","google/gemma-4-E4B-it", "google/gemma-4-31B-it", "google/gemma-4-26B-A4B-it"]

from transformers import AutoProcessor, AutoModelForMultimodalLM

model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f"Model loaded: {MODEL_ID}")

## 3. Document Converter

Handles all file types → JPEG image(s) saved to `converted/` folder.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
from typing import List, Optional

import fitz  # PyMuPDF
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

matplotlib.use("Agg")  # non-interactive backend for rendering

CONVERTED_DIR = "converted"
PDF_DPI = 200

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".gif", ".bmp", ".tiff", ".tif", ".webp", ".heic", ".heif"}
PDF_EXTENSIONS = {".pdf"}
SPREADSHEET_EXTENSIONS = {".xlsx", ".xls", ".csv", ".ods", ".xlsm", ".xlsb"}
OFFICE_EXTENSIONS = {".doc", ".docx", ".odt", ".ppt", ".pptx", ".rtf"}


def _ensure_dir(directory: str):
    Path(directory).mkdir(parents=True, exist_ok=True)


def _pdf_to_images(pdf_path: str, save_dir: str, stem: str) -> List[str]:
    """Convert each PDF page to a JPEG image using PyMuPDF."""
    doc = fitz.open(pdf_path)
    saved = []
    for i, page in enumerate(doc):
        pix = page.get_pixmap(dpi=PDF_DPI)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        out_path = os.path.join(save_dir, f"{stem}_page{i+1}.jpeg")
        img.save(out_path, "JPEG")
        saved.append(out_path)
    doc.close()
    return saved


def _image_to_jpeg(src_path: str, save_dir: str, stem: str) -> List[str]:
    """Convert any image format to JPEG."""
    out_path = os.path.join(save_dir, f"{stem}.jpeg")
    ext = Path(src_path).suffix.lower()
    if ext in {".jpg", ".jpeg"}:
        shutil.copy(src_path, out_path)
    else:
        with Image.open(src_path) as img:
            if img.mode not in ("RGB", "L"):
                img = img.convert("RGB")
            img.save(out_path, "JPEG")
    return [out_path]


def _spreadsheet_to_images(src_path: str, save_dir: str, stem: str) -> List[str]:
    """Render spreadsheet (xlsx/csv) sheets as table images using matplotlib."""
    ext = Path(src_path).suffix.lower()
    saved = []

    if ext == ".csv":
        sheets = {"sheet1": pd.read_csv(src_path)}
    else:
        xls = pd.ExcelFile(src_path)
        sheets = {name: xls.parse(name) for name in xls.sheet_names}

    for sheet_name, df in sheets.items():
        if df.empty:
            continue

        # Limit rows per image to keep text readable
        max_rows = 50
        chunks = [df.iloc[i:i+max_rows] for i in range(0, len(df), max_rows)]

        for chunk_idx, chunk in enumerate(chunks):
            n_rows, n_cols = chunk.shape
            fig_width = max(8, n_cols * 2.0)
            fig_height = max(3, (n_rows + 1) * 0.35)
            fig, ax = plt.subplots(figsize=(fig_width, fig_height))
            ax.axis("off")

            table = ax.table(
                cellText=chunk.astype(str).values,
                colLabels=chunk.columns.tolist(),
                cellLoc="center",
                loc="center",
            )
            table.auto_set_font_size(False)
            table.set_fontsize(8)
            table.scale(1.2, 1.4)

            # Style header row
            for (row, col), cell in table.get_celld().items():
                if row == 0:
                    cell.set_facecolor("#4472C4")
                    cell.set_text_props(color="white", fontweight="bold")

            suffix = f"_{sheet_name}" if len(sheets) > 1 else ""
            chunk_suffix = f"_part{chunk_idx+1}" if len(chunks) > 1 else ""
            out_path = os.path.join(save_dir, f"{stem}{suffix}{chunk_suffix}.jpeg")
            fig.savefig(out_path, format="jpeg", dpi=150, bbox_inches="tight", pad_inches=0.2)
            plt.close(fig)
            saved.append(out_path)

    return saved


def _office_to_images(src_path: str, save_dir: str, stem: str) -> List[str]:
    """Convert Office doc → PDF via LibreOffice → page images via PyMuPDF."""
    lo_cmd = shutil.which("libreoffice") or shutil.which("soffice")
    if not lo_cmd:
        raise RuntimeError(
            "LibreOffice not found. Install it to convert Office documents.\n"
            "  macOS:  brew install --cask libreoffice\n"
            "  Ubuntu: sudo apt-get install libreoffice"
        )

    # Convert to PDF first
    result = subprocess.run(
        [lo_cmd, "--headless", "--convert-to", "pdf", "--outdir", save_dir, src_path],
        capture_output=True, timeout=180,
    )
    if result.returncode != 0:
        raise RuntimeError(f"LibreOffice conversion failed: {result.stderr.decode(errors='ignore')}")

    pdf_path = os.path.join(save_dir, f"{stem}.pdf")
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"Expected PDF not found after conversion: {pdf_path}")

    # PDF → page images
    images = _pdf_to_images(pdf_path, save_dir, stem)
    os.remove(pdf_path)  # clean up intermediate PDF
    return images


def convert_document(file_path: str, output_dir: str = CONVERTED_DIR) -> List[str]:
    """
    Convert any supported document to JPEG image(s).

    Args:
        file_path: Path to the source document.
        output_dir: Directory to save converted images.

    Returns:
        List of paths to converted JPEG images.
    """
    src = Path(file_path)
    if not src.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    ext = src.suffix.lower()
    stem = src.stem
    _ensure_dir(output_dir)

    if ext in PDF_EXTENSIONS:
        converted = _pdf_to_images(str(src), output_dir, stem)
    elif ext in IMAGE_EXTENSIONS:
        converted = _image_to_jpeg(str(src), output_dir, stem)
    elif ext in SPREADSHEET_EXTENSIONS:
        converted = _spreadsheet_to_images(str(src), output_dir, stem)
    elif ext in OFFICE_EXTENSIONS:
        converted = _office_to_images(str(src), output_dir, stem)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

    print(f"Converted {src.name} → {len(converted)} image(s) in '{output_dir}/'")
    for p in converted:
        print(f"  → {p}")
    return converted


print("Converter ready.")

## 4. Set Input File(s) & Prompt

Set one or more **file paths** and your **prompt** below. Each file will be converted and all resulting images are sent together to the LLM.

In [ ]:
# ── Set your input file path(s) and prompt here ────────────────
# Single file:
#   INPUT_FILES = ["attachments/sample1.pdf"]
# Multiple files:
#   INPUT_FILES = ["attachments/sample1.pdf", "attachments/invoice.png", "attachments/data.xlsx"]

INPUT_FILES = [
    "attachments/sample1.pdf",
    # "attachments/sample2.png",
    # "attachments/data.xlsx",
]

PROMPT = "Extract all the key information from this document and return it as a structured JSON."  # @param {type:"string"}

# ── Convert all files ──────────────────────────────────────────
converted_paths = []
for f in INPUT_FILES:
    converted_paths.extend(convert_document(f))

print(f"\nTotal converted images: {len(converted_paths)}")

# Preview the converted images
images = [Image.open(p) for p in converted_paths]
n = len(images)
cols = min(n, 4)
fig, axes = plt.subplots(1, cols, figsize=(6 * cols, 8))
if cols == 1:
    axes = [axes]
for i, (ax, img) in enumerate(zip(axes, images[:4])):
    ax.imshow(img)
    ax.set_title(Path(converted_paths[i]).name, fontsize=9)
    ax.axis("off")
if n > 4:
    fig.suptitle(f"Showing 4 of {n} images", fontsize=10)
plt.tight_layout()
plt.show()

## 5. Send to Gemma 4 LLM

Sends the converted images to the model using the prompt set in Step 4.

In [ ]:
from transformers import TextStreamer


def send_to_llm(image_paths: list, prompt: str, enable_thinking: bool = True, max_tokens: int = 2048):
    """Load converted images and send to Gemma 4 with the given prompt."""
    images = [Image.open(p).convert("RGB") for p in image_paths]

    content = [{"type": "image"} for _ in images] + [{"type": "text", "text": prompt}]
    message = [{"role": "user", "content": content}]

    text = processor.apply_chat_template(
        message,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    inputs = processor(text=text, images=images, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    streamer = TextStreamer(processor)
    outputs = model.generate(**inputs, streamer=streamer, max_new_tokens=max_tokens)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    return processor.parse_response(response)


print(f"Sending {len(converted_paths)} image(s) to {MODEL_ID}...")
print(f"Prompt: {PROMPT}\n")

result = send_to_llm(converted_paths, PROMPT)

## 6. View Results

In [ ]:
for key, value in result.items():
    if key == "role":
        print(f"Role: {value}")
    elif key == "thinking":
        print(f"\n=== Thinking ===")
        print(value)
    elif key == "content":
        print(f"\n=== Response ===")
        print(value)
    elif key == "tool_calls":
        print(f"\n=== Tool Calls ===")
        print(value)
    else:
        print(f"\n{key}: {value}")